# EDA: stints and lineups (analytics/lineups)

Tables only (no matplotlib in the build environment). Default input is the synthetic fixture of `tests/_fx_lineups.py` (the local generator, whose rosters and rotations repeat lineups across seasons): **every number is fixture output, not a result**. On the Mac set `DATA_ROOT` to read the real section 12 stints and section 13 play-by-play (nothing under `heldout/` is read).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'analytics').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option('display.width', 200)
from analytics.lineups import load_inputs, SEASONS
from analytics.lineups.features import features
from analytics.lineups.validate import validate
from analytics.lineups import eda

DATA_ROOT = os.environ.get('DATA_ROOT')
inputs = load_inputs(DATA_ROOT is None, DATA_ROOT, SEASONS, seed=0, generator='local')
print('fixture output, not a result' if inputs['fixture'] else 'DATA_ROOT ' + DATA_ROOT)
stints = inputs['stints']
clean, counts = validate(stints)
print({k: v for k, v in counts.items() if k != 'per_season_rows'})
feats = features(stints, inputs['games'])

## Stint durations and the lineup_ok share

In [ ]:
print(eda.duration_summary(stints).round(2).to_string(index=False))

## Stint differential: points and per 100 possessions (possessions = duration_s / 28.8)

In [ ]:
print(eda.differential_summary(feats).round(2).to_string(index=False))

## Lineup possession counts: how many five-man lineups pass the 300-possession gate

In [ ]:
print(eda.lineup_possessions(feats, gate=300).round(2).to_string(index=False))
print(eda.lineup_possessions(feats, gate=100).round(2).to_string(index=False))

## Player minutes on court (top 10 per season, from stint durations)

In [ ]:
print(eda.player_minutes(stints, top=10).round(1).to_string(index=False))

## Correlation of raw lineup net rating across consecutive seasons

Lineups with at least 100 possessions in both seasons. A low correlation here is the ceiling any season-to-season lineup prediction faces.

In [ ]:
print(eda.cross_season_correlation(feats, min_poss=100).round(3).to_string(index=False))

## Stints rebuilt from the play-by-play substitutions

`analytics.lineups.stints.build_stints` derives period starters from the events (plus the starters table when present) and cuts at every substitution; on the fixture it reproduces the stints table where every player appears.

In [ ]:
from analytics.lineups.stints import build_stints, summarize
rebuilt = build_stints(inputs['playbyplay'], inputs['games'], inputs['starters'])
print(summarize(rebuilt))
m = rebuilt.merge(stints, on=['game_id', 'start_s', 'end_s'], suffixes=('', '_t'))
ok = m[m['lineup_ok']]
print('rows matched on (game_id, start_s, end_s):', len(m), 'of', len(stints), '; lineups agree on lineup_ok rows:', float((ok['home_ids'] == ok['home_ids_t']).mean()))